# Instacart Dataset Exploration

Let's look at what data we have and prepare a clean dataset for the recommendation model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path


ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
    
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"

print(f"Project root: {ROOT}")
print(f"Data raw: {DATA_RAW}")

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

Project root: c:\Users\Fama\instacart-recsys
Data raw: c:\Users\Fama\instacart-recsys\data\raw


## Loading the data

In [10]:
prior = pd.read_csv(DATA_RAW / "order_products__prior.csv", dtype={
    "order_id": "int32", "product_id": "int32", "add_to_cart_order": "int16", "reordered": "int8"
})
products = pd.read_csv(DATA_RAW / "products.csv")
aisles = pd.read_csv(DATA_RAW / "aisles.csv")
departments = pd.read_csv(DATA_RAW / "departments.csv")
orders = pd.read_csv(DATA_RAW / "orders.csv", dtype={
    "order_id": "int32", "user_id": "int32", "order_number": "int16", 
    "order_dow": "int8", "order_hour_of_day": "int8"
})

In [ ]:
print(f"prior: {prior.shape[0]:,} rows")
print(f"orders: {orders.shape[0]:,} orders")
print(f"products: {products.shape[0]:,} products")

## Quick look

In [12]:
prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [13]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [14]:
products.sample(10)

,product_id,product_name,aisle_id,department_id
35957,35958,Dairy Free Original Coconutmilk Coffee Creamer,91,16
21259,21260,Peanut Butter Cups,45,19
18973,18974,Lo Mein Egg Noodles,66,6
13117,13118,Organic Bosc Pear Bag,100,21
42990,42991,Kat Kit Disposable Tray,41,8
29363,29364,"Sweet Natural Cheddar Cheese, Milk Chocolate C...",100,21
40068,40069,"Soup Mix, Split Pea, with Seasonings",33,6
4754,4755,"Palmetto Cheese, Pimiento Cheese with Bacon",108,16
24795,24796,Organic Gluten Free Tapioca Flour,17,13
44790,44791,Dark Chocolate with Almonds,45,19


In [22]:
aisles.head()

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [23]:
departments.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


## Missing values

Quick check to see if there are any gaps

In [15]:
orders.isnull().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

In [16]:
prior.isnull().sum()

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

In [17]:
aisles.isnull().sum()

aisle_id    0
aisle       0
dtype: int64

In [18]:
departments.isnull().sum()

department_id    0
department       0
dtype: int64

In [19]:
products.isnull().sum()

product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

The NaNs in `days_since_prior_order` are normal, they're for first orders. Let's verify:

In [8]:
orders[orders["days_since_prior_order"].isna()]["order_number"].unique()

array([1], dtype=int16)

OK, they're all order_number=1. Replace with 0.

In [9]:
orders["days_since_prior_order"] = orders["days_since_prior_order"].fillna(0)

## Train/test split

In [10]:
orders["eval_set"].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

Keep only the prior for now

In [11]:
orders_prior = orders[orders["eval_set"] == "prior"].copy()

## Basic statistics

In [ ]:
n_users = orders_prior["user_id"].nunique()
n_products = products["product_id"].nunique()
n_orders = orders_prior.shape[0]

print(f"{n_users:,} users")
print(f"{n_products:,} products")
print(f"{n_orders:,} orders")

In [ ]:
# number of orders per user
orders_per_user = orders_prior.groupby("user_id").size()
orders_per_user.describe()

## Building the final dataset

In [ ]:
# merge step by step to manage memory
df = prior.merge(
    orders_prior[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]], 
    on="order_id"
)
del prior  # free RAM

df = df.merge(products[["product_id", "product_name", "aisle_id", "department_id"]], on="product_id")
df = df.merge(aisles, on="aisle_id")
df = df.merge(departments, on="department_id")

df.shape

In [21]:
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 14 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int32  
 1   product_id              int32  
 2   add_to_cart_order       int16  
 3   reordered               int8   
 4   user_id                 int32  
 5   order_number            int16  
 6   order_dow               int8   
 7   order_hour_of_day       int8   
 8   days_since_prior_order  float64
 9   product_name            object 
 10  aisle_id                int64  
 11  department_id           int64  
 12  aisle                   object 
 13  department              object 
dtypes: float64(1), int16(2), int32(3), int64(2), int8(3), object(3)
memory usage: 2.0+ GB


## Save

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PROCESSED / "prior_merged.csv", index=False)
print("done")